In [92]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [93]:
all_data = pd.read_csv('data.csv')
all_data

,seconds_std,add_to_cart_std,past_purchases_std,purchased,dataset
0,-2.0,-1.0,-1.5,0,train
1,-1.5,-0.5,-1.0,0,train
2,-1.0,-0.5,-0.5,0,train
3,-0.5,0.0,-0.5,0,train
4,0.0,0.0,0.0,0,train
5,0.5,0.0,0.0,1,train
6,1.0,0.5,0.5,1,train
7,1.5,0.5,0.5,1,train
8,2.0,1.0,1.0,1,train
9,-1.0,0.0,-1.0,0,train


In [94]:
train_data = all_data[all_data['dataset'] == 'train']
train_x = train_data.iloc[:,:3]
train_x = train_x.to_numpy()
train_y = train_data.iloc[:,3:4]
train_y = train_y.to_numpy()

train_x = torch.tensor(train_x,dtype=torch.float32,device='cuda')
train_y = torch.tensor(train_y,dtype=torch.float32,device='cuda')

In [95]:
class LinearModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = torch.nn.Linear(3,1,bias=True)

    def forward(self,x):
        return torch.sigmoid(self.linear(x))
model = LinearModel().to('cuda')

criterion = torch.nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.01,weight_decay=1e-4)

In [96]:
for epoch in range(1,1001):
    y_hat = model(train_x)
    loss = criterion(y_hat,train_y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    w1 = model.linear.weight[0, 0].item()
    w2 = model.linear.weight[0, 1].item()
    w3 = model.linear.weight[0, 2].item()
    b = model.linear.bias.item()
    print(f"epoch{epoch}   w1:{w1:.4f}   w2:{w2:.4f}   w3:{w3:.4f}   b:{b:.4f}   loss{loss.item():.4f}")

epoch1   w1:-0.5105   w2:-0.4097   w3:0.3435   b:-0.0965   loss1.0420
epoch2   w1:-0.5005   w2:-0.3997   w3:0.3535   b:-0.0865   loss1.0234
epoch3   w1:-0.4905   w2:-0.3898   w3:0.3635   b:-0.0765   loss1.0051
epoch4   w1:-0.4805   w2:-0.3798   w3:0.3735   b:-0.0665   loss0.9871
epoch5   w1:-0.4705   w2:-0.3698   w3:0.3835   b:-0.0566   loss0.9693
epoch6   w1:-0.4605   w2:-0.3598   w3:0.3934   b:-0.0468   loss0.9518
epoch7   w1:-0.4506   w2:-0.3499   w3:0.4034   b:-0.0369   loss0.9345
epoch8   w1:-0.4407   w2:-0.3400   w3:0.4133   b:-0.0272   loss0.9176
epoch9   w1:-0.4307   w2:-0.3301   w3:0.4232   b:-0.0175   loss0.9009
epoch10   w1:-0.4209   w2:-0.3202   w3:0.4331   b:-0.0079   loss0.8845
epoch11   w1:-0.4110   w2:-0.3104   w3:0.4429   b:0.0016   loss0.8684
epoch12   w1:-0.4011   w2:-0.3005   w3:0.4528   b:0.0109   loss0.8527
epoch13   w1:-0.3913   w2:-0.2908   w3:0.4626   b:0.0202   loss0.8372
epoch14   w1:-0.3815   w2:-0.2810   w3:0.4723   b:0.0293   loss0.8221
epoch15   w1:-0.371

In [97]:
with torch.no_grad():
    model.eval()
    y_pred = model(train_x)
    y_pred_binary = (y_pred > 0.5).float()
    accuracy = (y_pred_binary == train_y).float().mean().item()

print(accuracy)

1.0


In [98]:
test_data = all_data[all_data['dataset'] == 'test']
test_x = test_data.iloc[:, :3].to_numpy()
test_y = test_data.iloc[:, 3].to_numpy()

test_x = torch.tensor(test_x, dtype=torch.float32, device='cuda')
test_y = torch.tensor(test_y, dtype=torch.float32, device='cuda')

with torch.no_grad():
    pred = model(test_x)
    acc = ((pred > 0.5) == test_y).float().mean().item()
    print(f"✅ 测试集准确率: {acc:.4f}")  # 也应为 1.0

✅ 测试集准确率: 0.5200
